# 09 · Degradation probe and latency

**The probe** measures how much each model loses when a sensor goes dark, black or misaligned at test time (PH2). **Latency** times every model on one T4 at batch 1 in FP16, with parameters and GFLOPs (PH5).

| | |
|---|---|
| **Kaggle settings** | Accelerator **GPU T4 x2**, Internet **on** |
| **Inputs to attach** | `cffm-net-pilot`, raw LLVIP (or the output of 02), and the **saved outputs of notebooks 04, 05, 07 and 08** (*Add Input → Your Work*) |
| **Expected runtime** | about 1 to 1.5 hours on 1 T4 |
| **Produces** | `runs/probe/*/metrics.json`, `runs/latency/latency.json` |

In [ ]:
# Same setup cell in every notebook: find the project, install it, check the GPU.
import glob, shutil, subprocess, sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    PROJECT = Path("/kaggle/working/cffm-net-pilot")
    if not PROJECT.exists():
        # Kaggle mounts datasets at different depths under /kaggle/input, so search five levels.
        hits = [Path(p).parent for k in range(1, 6) for p in glob.glob("/kaggle/input" + "/*" * k + "/pyproject.toml")
                if (Path(p).parent / "src" / "cffm").is_dir()]
        # Earlier notebooks' outputs hold stale copies of the code, so prefer the uploaded dataset.
        hits.sort(key=lambda h: any((h.parent / d).exists() for d in ("runs", "data")))
        zips = [Path(p) for k in range(1, 5) for p in glob.glob("/kaggle/input" + "/*" * k + ".zip")
                if "cffm" in Path(p).name]
        if hits:
            shutil.copytree(hits[0], PROJECT, copy_function=shutil.copyfile)
        elif zips:                       # the zip was attached but not unpacked by Kaggle
            shutil.unpack_archive(str(zips[0]), "/kaggle/working")   # it holds one folder, cffm-net-pilot/
        assert PROJECT.exists(), "Add the 'cffm-net-pilot' dataset (the uploaded zip) as an input to this notebook."
        for p in [PROJECT, *PROJECT.rglob("*")]:     # inputs are read-only; our copy must be writable
            p.chmod(p.stat().st_mode | 0o200)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.171",
                    "faster-coco-eval>=1.6.7", "cloudpickle", "pytest"], check=True)   # cloudpickle: two-GPU launcher
    # Editable install, so DataLoader and DDP worker processes can import cffm too.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(PROJECT)], check=True)
else:
    PROJECT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "cffm").is_dir())
sys.path.insert(0, str(PROJECT / "src"))

import cffm
from cffm import env as cenv, pipeline
print("cffm", cffm.__version__, "imported from", Path(cffm.__file__).parent)   # must be inside PROJECT
env = cenv.setup()                       # also moves into the project folder
plan = pipeline.load_plan(env.project)   # configs/pilot.yaml

In [ ]:
pipeline.prepare("llvip", env, plan)
ckpts = pipeline.find_checkpoints(env)
for k, v in sorted(ckpts.items()):
    print(f"{k:24s} {v}")
missing = [r for r in plan["probe"]["runs"] if r not in ckpts]
assert not missing, f"attach the saved outputs of the notebooks that trained {missing}"

## 1. The probe

Each degradation is applied inside the forward pass with a fixed seed, so every model sees identical images.
*Flagged* drops also tell the model which sensor is missing, which only CFFM-Net can use.

In [ ]:
import pandas as pd
from cffm.train import probe

data = env.data / "llvip" / "data_paired.yaml"
rows = []
for run in plan["probe"]["runs"]:
    for kind in plan["probe"]["kinds"]:
        for flagged in ([False, True] if kind in plan["probe"]["flagged"] else [False]):
            m = probe(ckpts[run], data, kind, flagged=flagged, device=0, project=str(env.runs / "probe"))
            rows.append({"run": run, "probe": kind + (" (flagged)" if flagged else ""),
                         "AP": m["mAP50-95(B)"], "AP50": m["mAP50(B)"]})
            print(f"{run:20s} {rows[-1]['probe']:26s} AP {m['mAP50-95(B)']:.4f}")
probe_df = pd.DataFrame(rows)
probe_df.to_csv(env.runs / "probe" / "probe.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
piv = probe_df.pivot(index="probe", columns="run", values="AP").loc[probe_df["probe"].unique()]
ax = piv.plot(kind="barh", figsize=(10, 6), width=0.8)
ax.set_xlabel("COCO AP50-95 on LLVIP (full validation set)"); ax.invert_yaxis(); ax.grid(alpha=0.3, axis="x")
plt.tight_layout(); plt.show()

## 2. Parameters, GFLOPs and latency

GPU 0, FP16, batch 1, a 640 x 512 input, 50 warm-up and 300 timed passes with a short pause against throttling. TensorRT export is left to Phase 3.

In [ ]:
import json, torch
from cffm.train import _load, gflops, latency_ms
lat = {}
for run, w in sorted(ckpts.items()):
    m = _load(w)
    dual = getattr(m, "_dual", False)
    g = gflops(m, 640)
    t = latency_ms(m, imgsz=(512, 640), channels=4 if dual else 3, device="cuda:0", half=True, pause_s=0.02)
    lat[run] = {"params_M": sum(p.numel() for p in m.parameters()) / 1e6, "GFLOPs": g, **t}
    print(f"{run:24s} {lat[run]['params_M']:6.2f} M  {g:6.2f} GFLOPs  {t['median_ms']:6.2f} ms (p95 {t['p95_ms']:.2f})")
    del m; torch.cuda.empty_cache()
(env.runs / "latency").mkdir(parents=True, exist_ok=True)
(env.runs / "latency" / "latency.json").write_text(json.dumps(lat, indent=2))

In [ ]:
# /kaggle/working becomes the saved output, and the data is cheap to rebuild, so keep only the runs.
if env.platform == "kaggle":
    shutil.rmtree(env.data, ignore_errors=True)

### Keep the results

On Kaggle, click **Save Version → Save & Run All (Commit)** so `runs/` becomes an output that notebooks **09**
and **15** can attach (*Add Input → Your Work → this notebook*). Without a saved version the checkpoints vanish.